# MultiHeadAttention（多头自注意力机制层）
![self_attention](../../image/self_attention.png)  
![softmax](../../image/SoftMax函数.png)

In [1]:
import numpy as np

In [2]:
class MultiHeadAttention:
    def __init__(self, embed_dim, num_heads):
        print("开始初始化MultiHeadAttention")
        assert embed_dim % num_heads == 0, f"embed_dim {embed_dim} not divisible by num_heads {num_heads}"
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.W_q = np.random.randn(embed_dim, embed_dim) * 0.02
        self.W_k = np.random.randn(embed_dim, embed_dim) * 0.02
        self.W_v = np.random.randn(embed_dim, embed_dim) * 0.02
        self.W_out = np.random.randn(embed_dim, embed_dim) * 0.02
        print("初始化结果：")
        print("num_heads:", self.num_heads)
        print("head_dim:", self.head_dim)
        print("W_q矩阵（用来在自注意力模块里计算query矩阵）:", self.W_q.shape)
        print("W_k矩阵（用来在自注意力模块里计算key矩阵）:", self.W_k.shape)
        print("W_v矩阵（用来在自注意力模块里计算value矩阵）:", self.W_v.shape)
        print("W_out矩阵（用来在自注意力模块里处理输出）:", self.W_out.shape, "\n")

    def forward(self, x, mask=None):
        print("开始调用MultiHeadAttention-forward")
        batch, seq_len, d = x.shape
        print("batch,seq_len,model_dim:", batch, seq_len, d)

        Q = (x @ self.W_q).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        K = (x @ self.W_k).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        V = (x @ self.W_v).reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        print("query矩阵：", Q.shape)
        print("key矩阵：", K.shape)
        print("value矩阵：", V.shape)

        scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(self.head_dim)
        print("自注意力分数矩阵：", scores.shape)

        if mask is not None:
            print("mask矩阵：\n", mask)
            scores = scores + mask
        
        weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
        weights = weights / weights.sum(axis=-1, keepdims=True)
        print("softmax的输出结果：", weights.shape)
        attn_out = weights @ V
        print("self_attention的输出结果：", attn_out.shape, "\n ")

        attn_out = attn_out.transpose(0, 2, 1, 3).reshape(batch, seq_len, d)
        return attn_out @ self.W_out

In [3]:
if __name__ == "__main__":
    print("\n开始测试\n")
    # 超参数
    batch_size = 1
    seq_len = 5
    embed_dim = 16
    num_heads = 2

    # 实例化多头注意力
    mha = MultiHeadAttention(embed_dim=embed_dim, num_heads=num_heads)

    # 构造输入 x: [batch, seq_len, embed_dim]
    # 沿用前面Embedding输出，句子 I love building gpt models 共5个token
    x = np.random.randn(batch_size, seq_len, embed_dim) * 0.02

    # 构造GPT用的下三角mask（因果mask，看不到未来token）
    # mask 形状 [1,1,seq_len,seq_len]，未来位置填 -inf，softmax后权重趋近0
    mask = np.triu(np.full((seq_len, seq_len), -np.inf), k=1)
    mask = mask[np.newaxis, np.newaxis, :, :]

    # 前向传播
    out = mha.forward(x, mask=mask)

    print(f"输入 x shape: {x.shape}")
    print(f"输出 MHA shape: {out.shape}")
    print("\n第一个token的输出向量：")
    print(out[0,0])


开始测试

开始初始化MultiHeadAttention
初始化结果：
num_heads: 2
head_dim: 8
W_q矩阵（用来在自注意力模块里计算query矩阵）: (16, 16)
W_k矩阵（用来在自注意力模块里计算key矩阵）: (16, 16)
W_v矩阵（用来在自注意力模块里计算value矩阵）: (16, 16)
W_out矩阵（用来在自注意力模块里处理输出）: (16, 16) 

开始调用MultiHeadAttention-forward
batch,seq_len,model_dim: 1 5 16
query矩阵： (1, 2, 5, 8)
key矩阵： (1, 2, 5, 8)
value矩阵： (1, 2, 5, 8)
自注意力分数矩阵： (1, 2, 5, 5)
mask矩阵：
 [[[[  0. -inf -inf -inf -inf]
   [  0.   0. -inf -inf -inf]
   [  0.   0.   0. -inf -inf]
   [  0.   0.   0.   0. -inf]
   [  0.   0.   0.   0.   0.]]]]
softmax的输出结果： (1, 2, 5, 5)
self_attention的输出结果： (1, 2, 5, 8) 
 
输入 x shape: (1, 5, 16)
输出 MHA shape: (1, 5, 16)

第一个token的输出向量：
[ 2.52998490e-04  1.79394418e-05  4.23526898e-05  1.23278515e-04
  1.34293249e-04 -5.00499425e-05  5.41346945e-05 -2.73414677e-05
 -1.97694188e-05 -1.17357634e-05  3.33056018e-05 -4.15604741e-05
 -1.04484707e-04  4.67070771e-05  1.20087155e-04  2.90734047e-06]
